# Async Python

## A briefing on asynchronous python coding, essential in Agent engineering

Here is a masterful tutorial by you-know-who with exercises and comparisons.

https://chatgpt.com/share/680648b1-b0a0-8012-8449-4f90b540886c

This includes how to run async code from a python module.

### And now some examples:

In [6]:
# Let's define an async function

import asyncio

async def do_some_work():
    print("Starting work")
    await asyncio.sleep(4)
    print("Work complete")


In [2]:
# What will this do?

do_some_work()

<coroutine object do_some_work at 0x108047700>

In [3]:
# OK let's try that again!

await do_some_work()

Starting work
Work complete


In [4]:
# What's wrong with this?

async def do_a_lot_of_work():
    do_some_work()
    do_some_work()
    do_some_work()

await do_a_lot_of_work()

/var/folders/7x/wss496dn5qx9xsw6mrf_xqb80000gp/T/ipykernel_58061/1833959015.py:4: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()
/var/folders/7x/wss496dn5qx9xsw6mrf_xqb80000gp/T/ipykernel_58061/1833959015.py:5: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()
/var/folders/7x/wss496dn5qx9xsw6mrf_xqb80000gp/T/ipykernel_58061/1833959015.py:6: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()


In [7]:
# Interesting warning! Let's fix it

async def do_a_lot_of_work():
    await do_some_work()
    await do_some_work()
    await do_some_work()

await do_a_lot_of_work()

Starting work
Work complete
Starting work
Work complete
Starting work
Work complete


In [8]:
# And now let's do it in parallel
# It's important to recognize that this is not "multi-threading" in the way that you may be used to
# The asyncio library is running on a single thread, but it's using a loop to switch between tasks while one is waiting

async def do_a_lot_of_work_in_parallel():
    await asyncio.gather(do_some_work(), do_some_work(), do_some_work())

await do_a_lot_of_work_in_parallel()

Starting work
Starting work
Starting work
Work complete
Work complete
Work complete


In [9]:
import asyncio
import time

# 普通做法:傻等
def cook_slow():
    print("开始煮面...")
    time.sleep(3)        # 站在锅边干等3秒
    print("面好了!")
    print("开始烧水...")
    time.sleep(2)        # 又干等2秒
    print("水开了!")

# asyncio 做法:边等边干别的
async def cook_noodles():
    print("面下锅了,我去忙别的")
    await asyncio.sleep(3)   # "这里要等,我走啦!"
    print("面好了!")

async def boil_water():
    print("水壶放上了,我去忙别的")
    await asyncio.sleep(2)
    print("水开了!")

async def main():
    start = time.time()
    # gather = 同时推进多个任务
    await asyncio.gather(cook_noodles(), boil_water())
    print(f"总共用时: {time.time() - start:.1f} 秒")

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
import asyncio

async def steam_baozi(n):            # async def + yield = 异步生成器
    for i in range(1, n + 1):
        await asyncio.sleep(1)       # 【await 停】蒸1秒,期间让别的任务跑
        yield f"包子{i}"              # 【yield 停】递出去,等消费者要下一个

async def waiter():                  # 一个"别的任务",证明蒸包子时没人被卡住
    for _ in range(6):
        await asyncio.sleep(0.5)
        print("        (服务员在倒水...)")

async def main():
    async def eat():
        async for baozi in steam_baozi(3):   # 每圈: await __anext__()
            print("吃掉", baozi)

    await asyncio.gather(eat(), waiter())    # 老朋友 gather!

asyncio.run(main())

In [1]:
def make_baozi(n):              # 有 yield,所以这是生成器函数
    print("厨师系好围裙")        # 注意:调用时这行不会执行!
    for i in range(1, n + 1):
        print(f"  正在做第{i}个...")
        yield f"包子{i}"         # 递出去,定住
    print("收工!")

kitchen = make_baozi(3)
print(type(kitchen))            # <class 'generator'> ← 只拿到号码牌!
print("注意:厨师还没系围裙")

print(next(kitchen))            # 现在才开始执行!输出:系围裙、做第1个、包子1
print(next(kitchen))            # 从定住的地方继续:做第2个、包子2
print(next(kitchen))            # 包子3
# print(next(kitchen))          # 再要就抛 StopIteration("收工"后没了)

<class 'generator'>
注意:厨师还没系围裙
厨师系好围裙
  正在做第1个...
包子1
  正在做第2个...
包子2
  正在做第3个...
包子3


In [3]:
# for 循环其实就是不停地喊"下一个"
for baozi in make_baozi(3):
    print("吃掉", baozi)

厨师系好围裙
  正在做第1个...
吃掉 包子1
  正在做第2个...
吃掉 包子2
  正在做第3个...
吃掉 包子3
收工!


### Finally - try writing a python module that calls do_a_lot_of_work_in_parallel

See the link at the top; you'll need something like this in your module:

```python
if __name__ == "__main__":
    asyncio.run(do_a_lot_of_work_in_parallel())
```